In [1]:
import time
import yaml
import torch
import torch.nn as nn
import numpy as np
from pathlib import Path
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from torchmetrics.classification import BinaryF1Score, MulticlassAccuracy
from torcheval.metrics import Throughput
from argparse import ArgumentParser
import logging
from datetime import datetime

from qcm.model.reupload import HybridReuploadClassifier 
from qcm.model.reupload import QuantumHeadReupload
from qcm.data.datasets import get_dataloaders

from train_reupload import train_epoch

In [ ]:
def load_config(path: str) -> dict:
    with open(path, 'r') as f:
        return yaml.safe_load(f)
config = load_config('../configs/config_reupload.yaml')
dataset_type = config.get('dataset_type', 'pcam')

In [3]:
model = HybridReuploadClassifier(config=config, use_quantum=True)

In [4]:
train_loader, val_loader = get_dataloaders(config)

In [5]:
# Setup optimizer, scheduler, and loss function
optimizer = Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=config['training']['lr'])
scheduler = CosineAnnealingLR(optimizer, T_max=config['training']['epochs'])

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = HybridReuploadClassifier(config=config, use_quantum=True)
model.to(device)

HybridReuploadClassifier(
  (backbone): PCAMBackbone(
    (features): Sequential(
      (0): Conv2d(3, 2, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (1): BatchNorm2d(2, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
      (3): Conv2d(2, 4, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (4): BatchNorm2d(4, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): ReLU()
      (6): Conv2d(4, 8, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (7): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (8): ReLU()
      (9): AdaptiveAvgPool2d(output_size=(2, 2))
    )
    (latent): Sequential(
      (0): Linear(in_features=32, out_features=27, bias=True)
      (1): Tanh()
    )
  )
  (head): QuantumHeadReupload()
)

In [7]:
# Define run name and log file path
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
run_name = f"{dataset_type}_quantum_{timestamp}"
log_filepath = Path("./logs/training_log.csv")
log_filepath.parent.mkdir(exist_ok=True)

In [8]:
writer = SummaryWriter(log_dir=f"runs/test")
train_epoch_losses = []
val_epoch_losses = []

In [9]:
for epoch in range(config['training']['epochs']):
    avg_train_loss = train_epoch(model, train_loader, optimizer, scheduler, epoch, config, writer)
    # avg_val_loss = validate(model, val_loader, epoch, config, writer)
    
    train_epoch_losses.append(avg_train_loss)
    # val_epoch_losses.append(avg_val_loss)

2025-10-03 10:58:28,413 - INFO - Starting training epoch 0


KeyboardInterrupt: 

In [ ]:
train_loader.dataset = train_loader.dataset

ValueError: pic should be 2/3 dimensional. Got 4 dimensions.